# SD 1.5 LoRA Fine-Tuning — Colab Quickstart

End-to-end LoRA fine-tuning of Stable Diffusion 1.5, plus base-vs-LoRA CLIP-score evaluation. Runs on a free Colab T4 in ~45 minutes total.

**Before you start:**
1. Set runtime to GPU: **Runtime → Change runtime type → T4 GPU → Save**.
2. Run cells top to bottom (Shift+Enter). Don't use **Run all** — if a cell fails, you want to know which one.
3. Don't close the tab or idle for >90 min; Colab will disconnect.
4. Step 9 saves your results to Google Drive — **don't skip it**, or you lose everything on disconnect.

## Step 1 — Clone the repo

In [ ]:
%cd /content
!rm -rf sd-lora-finetune
!git clone https://github.com/kishoremadanagopal/sd-lora-finetune.git
%cd sd-lora-finetune

## Step 2 — Install dependencies (~3 min)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q xformers
!pip install -q --upgrade "torchao>=0.16.0"

## Step 3 — Verify the GPU is on
Should print `CUDA available: True` and `Device: Tesla T4`. If it says CPU, fix the runtime type and re-run.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Step 4 — Train the LoRA (~25–35 min)

Trains for 500 steps on the Pokémon dataset. Look for a progress bar like `train: 50%|▌| 250/500 ...`.

Lots of yellow warnings will scroll past — that's normal. Only worry if it ends with `Traceback`.

In [ ]:
!python scripts/train_lora.py --config configs/default.yaml \
  training.max_train_steps=500 \
  training.validation_every=250 \
  training.checkpoint_every=250 \
  dataset.max_train_samples=400

## Step 5 — Find the latest LoRA checkpoint
Should print a path ending in `lora_step_000500.safetensors` (or `_000250` if you stopped early).

In [ ]:
import glob, os
ckpts = sorted(glob.glob('outputs/checkpoints/lora_run/lora/lora_step_*.safetensors'))
assert ckpts, 'No checkpoint found — did training finish?'
latest = ckpts[-1]
print('Latest:', latest)
print('Size:', round(os.path.getsize(latest) / 1024, 1), 'KB')

## Step 6 — Evaluate (~5–8 min)

Generates 8 images with the base model and 8 with the LoRA model on the same prompts + same seed, then computes CLIP score for each.

Success looks like a final line: `[eval] mean CLIP - base: XX.XX | lora: XX.XX` with two **different** numbers. If the numbers are identical, the LoRA didn't load — check Step 1 cloned the latest code.

In [ ]:
!python scripts/evaluate.py --lora {latest} --prompts-file configs/eval_prompts.txt

## Step 7 — Show before/after comparison grids
Left half = plain SD 1.5. Right half = your fine-tuned LoRA. Same prompts, same seeds.

In [ ]:
from PIL import Image
import glob
from IPython.display import display
for p in sorted(glob.glob('outputs/eval/comparison/*.png'))[:4]:
    print(p)
    display(Image.open(p))

## Step 8 — Print the CLIP-score report
Per-prompt scores and means. Useful for filling in the README's Results table.

In [ ]:
import json
print(json.dumps(json.load(open('outputs/eval/report.json')), indent=2))

## Step 9 — Save everything to Google Drive (don't skip)

Saves your LoRA checkpoints, comparison images, base/lora images, and report to `MyDrive/sd_lora_results/`. You'll be prompted to authorize Drive access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/sd_lora_results
!cp -r outputs/eval /content/drive/MyDrive/sd_lora_results/
!cp outputs/checkpoints/lora_run/lora/*.safetensors /content/drive/MyDrive/sd_lora_results/
print('Saved to Google Drive -> MyDrive/sd_lora_results/')